# Game Overview

*Andrew Wang · 2026-08-08*

## Contents

1. [Introduction](#introduction)
2. [Game Mechanics](#game-mechanics)
3. [Design Choices](#a-single-game)
    * [Data Storage](#data-storage)
    * [Reproducibility](#reproducibility)
4. [Outcome Distributions](#outcome-distributions)
5. [Conclusion](#conclusion)

## Introduction

This is an investigation of simple risk-taking game that turned out to be surprisingly rich. The game is commonly known as Balloon Analogue Risk Task (BART) and is often used in psychology to measure different people's intuitive approaches and habits to risk-taking. The game is about inflating balloons of different colors and different likelihoods of popping without popping them. In each turn, the player is asked to the decide whether to pump or cash; pumping would increase the value of the balloon and cashing would safely store the current value of the balloon as points. Of course, the goal is to maximize the number of points after a certain finite number of balloons. To clarify, each balloon has a different color, and each balloon's popping probability is dependent on its color. Additionally, the popping probabilities are constant (i.e the second pump is not more likely to burst the balloon than the first). When I was first introduced to this game, intuitively I thought it would be smart to spend the first 10-20 balloons always pumping to figure the ballpark estimates of the popping probabilities of the balloons. Then, I would model the remaining balloon popping times as geometrically distributed random variables and hence there would be an optimal policy to maximize the expected value. This strategy would later develop into the explore-then-exploit (also known as $\epsilon$-greedy) strategy which will be discussed later. Other candidates of strategies I thought of included for example always pumping once, since you would expect the popping probability to be $\frac{1}{2}$ and would in expectation take 2 pumps to pop. I first encountered this game in a setting similar to a browser-based computer game, and to try different strategies without physically having to play them I had to first develop the mechanics of the game in python (which at that time was my preferred programming language / now it's unfortunately java after taking cs61b at berkeley). 

After exploring the game in-depth, I discovered that it roughly resembled a famous math problem known as multi-armed bandits, which is also highly relevant in reinforcement learning. And that, the balance of exploring balloon colors and exploiting them was a difficult and well-researched problem. After developing efficient strategies for the original version of the game, I wanted to see what would happen if I extended the game to a multiplayer version, and whether the "optimal" strategies from before was still optimal, or that they could be exploited by some best-response strategies that specifically are designed for them. It was interesting designing the specific mechanics of the multiplayer extension of the game, as many different versions were definately possible and appealing. I settled with a version that somewhat resembles an auction or quoting prices for an option where you want to always quote better prices than your opponent, but never higher than the true value. Moreover, I was curious what strategies RL agents would develop by playing against the different optimal strategies as well as playing themselves repeatedly. 

Another aspect to this project I quickly learnt was both the importance of design choices and their consequences in performance and latency, but more importantly, efficient ways to measure and improve performance. The first iteration of the project had many flawed design choices despite having actually thought through them before. Interestingly though, after carefully profiling the latency of different aspects of the program, the bottleneck was completely different from what I expected. This demonstrated to me the importance of careful profiling, as the latency profile of a program can be very counterintuitive. In the second iteration of the project, I improved many flaws and weaknesses, and most importantly the time it takes to run it. Optimizing for time wasn't just for fun either, many of my initial results were highly noisy and to reduce the variation I would have to run the same game multiple times and take the average. However, without an efficiently running program, this would take ages. 

## Game mechanics

The game consists of 3 layers of hierarchy: turns, balloons, players. Each player makes decisions for each balloon, and for each balloon, they have to make a decision to pump or cash every turn until the balloon pops or they cash. Naturally, 

## Design Choices

The most challenging part of this (or any project) was to make the initial design choices. The philosophy followed was to try to focus as little as possible to instantly find the _optimal_ way to do something, but rather find a working solution first and then improve. However, as no philosophy is perfect, this led to many chunks of code having to be completely rewritten instead of _improved_ as the first iteration of code not being scalable.

### Data Storage

Naturally, we want to record _useful_ data from each round in each game in an efficient manner to later analyze. Obvious examples of columns we should record are __turn, balloon_number, balloon_color, balloon_value, strategy_name, player_action, popped, player_unrPnL, player_PnL__. This can be seen in the earliest fully working git commits (for example: 2047c5fc108562d4e40e9df61a1b630d9eeda669). The most natural way to handle tabular data like this in python was to use _pandas_. However, I quickly realized that since we need to append one row at a time after each turn, and the concatenation function in pandas rewrites the entire table (so it is roughly $O(n^2)$ asymptotically timewise, where $n$ is the number of turns). Then, I switched to using SQL as appending a row is $O(1)$. This is the initial design the came with the first fully-functioning version of the project (as in the earlier commit SHA). Appending rows to an SQL table works the following way. First, we need to establish an SQLite connection. Then, we make our changes, and finally, we have to commit our changes and close the connection. Importantly, we cannot access the changes in the SQL database through another connection if we don't commit and close the initial connection. 

By simulating a few games with a few strategies, I quickly noticed that with around $100$ balloons and $4$ strategies, each game took a few seconds. This was surprising to me and I decided to quickly profile where exactly the bottleneck was. I first tested how long exactly, the entire pipeline took using total_time.py:

```jsonl
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "total_time", "trial": 0, "wall_time": 3.4734380000736564, "cpu_time": 1.890625, "date": "2026-08-06 01:22:38"}
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "total_time", "trial": 1, "wall_time": 3.542228600010276, "cpu_time": 2.03125, "date": "2026-08-06 01:22:42"}
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "total_time", "trial": 2, "wall_time": 3.1841210001148283, "cpu_time": 1.71875, "date": "2026-08-06 01:22:45"}
```

Note that here wall_time is the total time used, and cpu_time is time where the cpu was active processing. This was with $150$ balloons, which I definately should have noted when I ran these tests. This confirmed my suspicions: the pipeline was taking a lot longer than it should be. Importantly, this meant that since I would have to run games $100$s of times in my data analysis to reduce variance, this time inefficiency would have to improve ASAP. I also checked precisely the scaling effect on the time using scaling.py:


```jsonl
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "scaling", "n_size": 50, "wall_time": 1.2350743000861257, "ms_per_turn": 2.5154262730878325, "date": "2026-08-06 01:23:17"}
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "scaling", "n_size": 100, "wall_time": 2.3438203998375684, "ms_per_turn": 2.164192428289537, "date": "2026-08-06 01:23:20"}
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "scaling", "n_size": 200, "wall_time": 5.262789099942893, "ms_per_turn": 2.4523714352017207, "date": "2026-08-06 01:23:25"}
{"commit": "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe", "script": "scaling", "n_size": 400, "wall_time": 10.190721800085157, "ms_per_turn": 2.631901291344307, "date": "2026-08-06 01:23:35"}
```

I'm remarking this since often optimizing a program WITHOUT your task really requiring it is pretty much a waste of time. The next step was to check at which steps exactly, the time was spent. Another important note to make is that the ms_per_turn stays about constant despite increasing the number of balloons exponentially. This signifies a good thing: even this slow pipeline is __NOT__ _algorithmically slow_ (i.e it is $O(n)$ where $n$ is the number of balloons).

Using the profiling library cProfile (which, as the name suggests, runs in C to minimize additional time due to running the test itself), and running profiling.py, I obtained the following result:

In [7]:
import pstats
from pathlib import Path

PROF_DIR = Path.cwd().parents[1] / "profiling" / "results"
p = pstats.Stats(str(PROF_DIR / "f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe_150.prof"))
p.strip_dirs() 
p.sort_stats("tottime").print_stats()


Thu Aug  6 01:23:03 2026    c:\Users\AW060\OneDrive\Quant\Summer 26\Projects\BalloonGame\profiling\results\f5cfedca8e77ad7c2b2823c0df0699be8a2adfbe_150.prof

         29031 function calls in 3.650 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1489    2.163    0.001    2.163    0.001 {method 'commit' of 'sqlite3.Connection' objects}
     3143    0.985    0.000    0.985    0.000 {method 'execute' of 'sqlite3.Cursor' objects}
      497    0.378    0.001    0.593    0.001 simple_strategies.py:83(posterior)
      499    0.020    0.000    0.020    0.000 {method 'close' of 'sqlite3.Connection' objects}
      450    0.017    0.000    3.643    0.008 Classes.py:164(per_balloon_loop)
      323    0.015    0.000    0.015    0.000 {method 'fetchall' of 'sqlite3.Cursor' objects}
     1331    0.010    0.000    0.010    0.000 {method 'fetchone' of 'sqlite3.Cursor' objects}
      497    0.008    0.000    0.008    0.000 simple_strateg

Surprisingly, most of the time is spent on SQL commits. Note also that this function was called a large number of times since we had to commit after every turn. The reason we did this was so that other SQL connections could access the newest information possible. One may argue that this feature is a bit unnecessary as we don't really need to see the SQL table for data analysis mid-game. However, since strategies need to have access to all information from the past to make informed choices of what actions to take, this was indeed necessary (i.e the strategies need access to the SQL database mid-game). From here we needed to figure out a game design that could solve this problem efficiently.

The underlying requirement for any solution was that it cannot commit to the SQL database after every turn, and optimally, we need to minimize the number of commit. The solution I arrived with relies on the following idea. At each turn, strategies does not need to have access to the entire history of balloons, but only its _beliefs_ about the balloon colors prior to seeing the current balloon. Then, it would only have to adjust its beliefs after seeing the current balloon. In a way, at each turn, the strategies digest the new information into a few key updates on its beliefs, and hence need not keep track of the entire history. To be specific, what I mean by beliefs here is how many times the strategy would pump a certain colored balloon if the current balloon is of that color. In this way, we only need to commit once in the end, as strategies no longer need to access SQL database mid-game. 

Moreover, this kind of belief-updating design naturally inspired a slightly new design of how strategies worked too. Strategies now need a method that updates its belief, which is called after each balloon. Moreover, it asks the question of what kinds of information about balloons it needs to collect to apropriately update its beliefs. To make the engine scalable, I decided also to introduce Observation objects, which would carry all the information that strategies would need to observe in order to update its beliefs. Another requirement that I kept in mind when designing this next iteration of the project was the ambition to extend the game into a multiplayer-version in which two players simultaneously make moves. The exact rules will be explained in the section about the multiplayer version of the game, but the important part I wanted to note was that the multiplayer version with only 1 strategy entered should reduce to the game we are familiar with. In other words, the design we arrive at must allow an easy way to extend into a mutliplayer version. The design I arrived at can be summarized by the following diagram: 

In [8]:
from IPython.display import Markdown, display
display(Markdown(open("../../design/engine.md").read()))





# balloon_loop()

``` mermaid
flowchart TD
balloon["input = [balloon: Balloon]"]
player_loop{"For every strategy"}
action["strategy.action()"]
thresholds["thresholds: dict"]
resolve_round["resolve_round()"]
update_belief["strategy.update_belief()"]
sql_insert["sql_insert()"]

balloon --> player_loop
player_loop -- balloon: Balloon --> action -- threshold: int --> thresholds -- thresholds: dict[str, int] --> resolve_round -- obs_dict: dict[str, Observation] --> update_belief
resolve_round -- sql_list: list[tuple] --> sql_insert
player_loop -- balloon: Balloon --> resolve_round

```

After implementing this design, I ran the same latency profiling tests and achieved a significant improvement. Firstly, the total time was reduced from $\approx 4$ seconds to $\approx 0.05$ seconds.

```jsonl
{"commit": "04487f7db3409aa5cbe18dd1aafd62087d646df5", "script": "total_time", "trial": 0, "wall_time": 0.07963699987158179, "cpu_time": 0.046875, "date": "2026-08-08 10:26:07"}
{"commit": "04487f7db3409aa5cbe18dd1aafd62087d646df5", "script": "total_time", "trial": 1, "wall_time": 0.06458579981699586, "cpu_time": 0.046875, "date": "2026-08-08 10:26:07"}
{"commit": "04487f7db3409aa5cbe18dd1aafd62087d646df5", "script": "total_time", "trial": 2, "wall_time": 0.06787540018558502, "cpu_time": 0.015625, "date": "2026-08-08 10:26:07"}

```

Specifically, going from an average time of $3.399$ seconds to an average of $0.0707$ seconds is a $\approx 48$ times improvement. I was also curious, now that I'm only commiting once per game, what the new bottleneck would be. Running profiling.py again I obtained the following: 

In [9]:
import pstats
from pathlib import Path

PROF_DIR = Path.cwd().parents[1] / "profiling" / "results"
p = pstats.Stats(str(PROF_DIR / "04487f7db3409aa5cbe18dd1aafd62087d646df5_150.prof"))
p.strip_dirs() 
p.sort_stats("tottime").print_stats()


Sat Aug  8 10:36:12 2026    c:\Users\AW060\OneDrive\Quant\Summer 26\Projects\BalloonGame\profiling\results\04487f7db3409aa5cbe18dd1aafd62087d646df5_150.prof

         26636 function calls (26632 primitive calls) in 0.099 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      912    0.038    0.000    0.038    0.000 {method 'execute' of 'sqlite3.Cursor' objects}
       12    0.010    0.001    0.010    0.001 {method 'commit' of 'sqlite3.Connection' objects}
        6    0.010    0.002    0.019    0.003 engine.py:102(initialization)
        6    0.009    0.001    0.076    0.013 engine.py:132(balloon_loop)
      900    0.003    0.000    0.003    0.000 {method 'reduce' of 'numpy.ufunc' objects}
      900    0.002    0.000    0.007    0.000 fromnumeric.py:66(_wrapreduction)
      300    0.002    0.000    0.002    0.000 strategies.py:128(thompson_sampler)
      300    0.002    0.000    0.006    0.000 strategies.py:115(update_beliefs

Interestingly, the most time-heavy functions are still the ones related to SQL handling; perhaps in the future, I will look into a better storage system. However, it is relatively still the most time-inefficient part of the program, the program in its entirety is now sufficiently fast to the extent of which further optimization would not be necessary.

### Reproducibility

The next important design choice covers the issue of reproducibility. The goal is that, although games are heavily reliant on randomness, we should be able to re-run any particular game, and obtain the same results. The standard way to handle this is the use of seeds; although it sounds simple, as we will see, coming up with such a design is far from trivial.

Particularly, there are the following features/requirements that we desire from our design:
1. player_a vs player_b should get the same result as player_b vs player_a (i.e the order in which the user enters the strategies should not matter)
2. Same seed + same strategies -> same results.
3. If multiple games a run together under the same seed (round-robin type experiment), previous games should not affect the current games. For example, the result of the player_b vs player_c match should be the same regardless if we run the games player_a vs player_b then player_b vs player_c, or player_d vs player_e then player_b vs player_c.
4. If multiple games are run under the same seed, each individual game should obtain its own seed, and running that specific game with that specific seed should obtain the same result.

Note: the user will only enter a single seed no matter how many games they will run. Each game that is run on will receive their corresponding game seed as the name of the name of the csv file where the data for that specific game is stored. 

Before designing how we are going to use seeding, we must clarify _all_ sources of randomness in our pipeline since those are exactly the areas where seeding is necessary. Sources of randomness are the following:

1. Balloon colors
2. Balloon popping times given the colors 
3. Strategies' use of ranomness

Note that we want the popping times to be the same for each strategy on the same seed. An example will demonstrate this more clearly. 

Suppose the seed generates the following balloons:

[red, blue, green, purple]

Now, when strategy_a (always pump) plays in this seed (single player) the popping times are the following:

[1, 3, 4, 3]

Now, when strategy_b plays the same seed, although the colors of the balloons don't specify exact popping times, the popping times should be the same for fairness and reproducibility. This means when strategy_b plays the popping times (if they ever pop) will be 

[1, 3, 4, 3]

as well.

In other words, the seed needs to first produce a sequence of balloon colors. Given these colors, we must sample their popping times from a geometric distribution with 1 - each color's popping probability as the parameter.

Moreover, some strategy like Thompson Sampling utilizes randomness within the strategy (samples from a beta distribution). This part must also be reproducible. 

Each strategy will receive their own RNG object for randomness within the strategy, and each game is equipped with its own game RNG object. Since Game objects contain the strategies which contain RNG objects, build_games() only need to return Game objects. Our design can be summarized by this diagram:

In [6]:
from IPython.display import Markdown, display
display(Markdown(open("../../design/reproducibility.md").read()))





```mermaid 
flowchart TD
master_seed["Master seed"]
seeds["Generate n balloon seeds [s_1, s_2...s_n]"]
seeds_mult["Generate kC2x2n seeds [s_1, s_2...s_kC2x2n] for each strategy in each pair, where k is number of strategies"]
strat_sort["Sort strategies"]
Instance_strat["Instantiate strategies with their RNG objects"]
Instance_games["Instantiate Game objects with their RNG objects"]
multiplayer["multiplayer = ?"]
save["append Game object to list"]
seeds_single["Generate nk seeds for each strategy in each seed, where k is number of strategies"]

master_seed --> strat_sort


strat_sort --> seeds --> multiplayer -- 1 --> seeds_mult

multiplayer -- 0 --> seeds_single --> Instance_strat

Instance_games --> save --> Instance_strat
seeds_mult --> Instance_strat --> Instance_games


```

Importantly, the nk seeds generated after the multiplayer split, are generated using the balloon seeds generated in the earlier step. In this way, individual games are completely reproducible by inserting the game seed (which the csv file will be named after) as the master seed and the number of seeds as $1$. 

### Scalable Strategy Design


Another goal of this project is that it should be possible to extend it. Whether this means adding more strategies or running additional experiments or adding/removing censorship, it should be easily done by the user. Specifically, an important aspect of the game is how much information each strategy receives after each round (i.e true popping time, opponent's threshold, etc). The design should allow easy manipulation of what kind of information is released to the strategies. 

Moreover, it should be easy for the user to create their own strategies. More importantly, they should be able to quickly test their strategies. 

Another important point is that the parser should recognize which strategies exist and which don't (i.e entering a name of strategy that don't exist in the codebase should be easily detectable). Similarly, parameters of strategies should be easily verifiable. This means that if a parameter takes integers, the parser should know this, and if the user inserts a string, this should be easily detectable.

Our Strategy design can achieve these goals by the following measures:
1. Every strategy is a subclass of the Strategy parent class, which contains the register of every strategy that exists (and their respective classes).
2. User writing a strategy automatically initializes it in the aforementioned register, which allows the parser to check which strategies exist.
3. Parameter types are specificied in the definition of strategies as an attribute of the strategy. This allows the parser to easily verify parameters inputted by the user.
4. context and Observation objects allow easy manipulation for the user to add additional requirements of strategies as well as changing censorship.

This design can be summarized with the following diagram:

In [1]:
from IPython.display import Markdown, display
display(Markdown(open("../../design/strategy.md").read()))




This is the parent class for every strategy (default strategies and custom user-made strategies). When user defines their own strategies, they have to specify the strategy's KEY and PARAMS if they have any. Each parameter in PARAMS list is in the following form:

(attr_name, type)

For example, ("pump_value", int)

REGISTER is a dictionary that maps strategy keys to the apropriate class. 

At initialization of subclasses (custom strategies) it checks if its key is in REGISTER, and adds key-value pair if it does not contain it already. If it already contains the key, an exception is raised informing of a duplicate strategy. 

from_name() returns a STRATEGY INSTANCE given the parameters, class, and context. 
```mermaid
flowchart LR
attr["Attributes"]
args["Arguments"]
methods["methods"]
params["PARAMS: list[tuple]"]
register["REGISTER: dict[str, cls]"]
key["KEY: str"]
name["name: str"]
ctx["ctx: Context"]
rng[rng: RNG object]
Strategy["Strategy"]
init["__init_subclass__(cls, **kwargs)"]
from_name["from_name(cls, name: str, ctx: dict)"]

Strategy --> attr
Strategy --> args
Strategy --> methods


args --> name & ctx & rng
attr --> params & key & register
methods --> init & from_name

```


The Context object is designed to include additional information that strategies might need. This could for example include the number of balloons (epsilon strategies use this), as well as all the available colors (which is used at initialization of strategies). We made it an object so that users in the future can add additional features that more sophisticated strategies would need. Note that this Context object will be shared between several strategies, so should be treated as an immutable object.

## Outcome Distributions

## Conclusion